# 函数定义

In [1]:
import argparse
import textwrap
import os
import time
import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool
from livelossplot import PlotLosses
from sklearn.metrics import f1_score
import shutil
import pandas as pd

/home/admin123/software/anaconda3/envs/dgl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def prepare_data(dataset, shuffle=False, prog_args=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    return dgl.dataloading.GraphDataLoader(
        dataset,
        batch_size=prog_args.batch_size,
        shuffle=shuffle,
        num_workers=prog_args.n_worker,
    )


def train(dataset, model, prog_args, val_dataset=None):
    """
    training function
    """
    # 初始化模型存储路径
    if not os.path.exists(prog_args.save_dir + "/" + prog_args.dataset):
        os.makedirs(prog_args.save_dir + "/" + prog_args.dataset)
    else:
        shutil.rmtree(prog_args.save_dir + "/" + prog_args.dataset)
        os.makedirs(prog_args.save_dir + "/" + prog_args.dataset)

    f_train_log = open(prog_args.save_dir + "/" + prog_args.dataset + "/train_log.csv", 'w')
    temp_line = 'epoch, accuracy, F1 score'
    line_flag = 3
    if prog_args.family_fold_type == 'GTA':
        for x in range(1, 10):
            temp_line = temp_line + f', Class {x} F1 score'
            line_flag += 1
    elif prog_args.family_fold_type == 'GTB':
        for x in range(1, 11):
            temp_line = temp_line + f', Class {x} F1 score'
            line_flag += 1
    else:
        raise ValueError(f"Invalid family_fold_type: '{prog_args.family_fold_type}'. Valid options are 'GTA' and 'GTB'.")
    temp_line = temp_line + '\n'
    f_train_log.write(temp_line)

    # 检查存储路径
    dir = prog_args.save_dir + "/" + prog_args.dataset
    if not os.path.exists(dir):
        os.makedirs(dir)
    
    dataloader = dataset
    optimizer = torch.optim.Adadelta(filter(lambda p: p.requires_grad, model.parameters()))  # 初始化优化器

    if prog_args.cuda > 0:
        torch.cuda.set_device(0)
    
    for epoch in range(0, prog_args.epoch):
        # 暂时存储，方便计算准确性
        train_pred_indi = torch.tensor([], device='cuda')
        train_label_indi = torch.tensor([], device='cuda')
        begin_time = time.time()
        model.train()
        accum_correct = 0
        total = 0
        print("\nEPOCH ###### {} ######".format(epoch))
        computation_time = 0.0
        for batch_idx, (batch_graph, graph_labels) in enumerate(dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            model.zero_grad()
            compute_start = time.time()
            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            train_pred_indi = torch.cat((train_pred_indi, indi), dim=0)
            train_label_indi = torch.cat((train_label_indi, graph_labels), dim=0)
            correct = torch.sum(indi == graph_labels).item()
            accum_correct += correct
            total += graph_labels.size()[0]
            loss = model.loss(ypred, graph_labels)
            loss.backward()
            batch_compute_time = time.time() - compute_start
            computation_time += batch_compute_time
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=prog_args.clip)
            optimizer.step()
        
        train_f1_score = f1_score(train_pred_indi.cpu(), train_label_indi.cpu(), average='macro')
        temp_f1_score = f1_score(train_pred_indi.cpu(), train_label_indi.cpu(), average=None)
        train_f1_score_class_dict = {}
        train_f1_score_class_keys = []
        for i in range(len(temp_f1_score)):
            # 生成与原变量名相同的key，如"train_f1_score_class_1"
            key = f"train_f1_score_class_{i+1}"
            # 存储对应的值（i是列表索引，i+1是类别编号）
            train_f1_score_class_keys.append(key) # 列表是有先后顺序的，保证写入log的时候不会乱序
            train_f1_score_class_dict[key] = temp_f1_score[i]
        train_accu = accum_correct / total

        elapsed_time = time.time() - begin_time

        if prog_args.save_dir is not None:
            torch.save(
                model.state_dict(),
                prog_args.save_dir
                + "/"
                + prog_args.dataset
                + "/model.iter-"
                + "{:04d}".format(epoch)
            )

        temp_line = f'{epoch}, {train_accu * 100}, {train_f1_score}'
        temp_line_flag = 3
        for x in train_f1_score_class_keys:
            temp_line = temp_line + f', {train_f1_score_class_dict[x]}'
            temp_line_flag += 1
        temp_line = temp_line + '\n'
        assert temp_line_flag == line_flag, 'Wrong log line number, please check what happen.'
        f_train_log.write(temp_line)

        print("train accuracy for this epoch {} is {:.2f}%".format(epoch, train_accu * 100))
        print("loss {:.4f} with epoch time {:.4f} s & computation time {:.4f} s ".format(loss.item(), elapsed_time, computation_time))

        torch.cuda.empty_cache()
    f_train_log.close()
    return 'Trian successfully'



# 初始化状态

In [3]:
fold_num = 1
family_fold_type = 'GTB'
print("{:=^100}".format(f'fold num is : {fold_num}, family type is : {family_fold_type}'))

print("{:=^100}".format('prog_args'))
prog_args = argparse.Namespace(dataset=f'GTmining_6_6_{family_fold_type}_fold{fold_num}', pool_ratio=0.10, num_pool=1, cuda=1, lr=1.0, clip=float("inf"),
                               batch_size=128, epoch=10, n_worker=10, gc_per_block=3, aggregator_type="meanpool",
                               dropout=0.00, method="diffpool", bn=True, bias=True, save_dir=f"./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type=family_fold_type)
print( textwrap.fill(str(prog_args), width=100))

print("{:=^100}".format('加载数据'))
dataset_train = tu.RhaFinderDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/train/')
dataset_validation = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/validation/')
dataset_test = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/test/')
dataset_nova = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}/fold{fold_num}/nova/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)
nova_dataloader = prepare_data(dataset_nova, shuffle=False, prog_args=prog_args)


input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
input_dim_nova, label_dim_nova, max_num_node_nova = dataset_nova.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test, max_num_node_nova])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test)+ len(dataset_nova))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

if family_fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif family_fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError(f"Invalid family_fold_type: '{prog_args.family_fold_type}'. Valid options are 'GTA' and 'GTB'.")
df_cluster = pd.read_excel(f'../data/cluster/{family_fold_type}/dataseat_split_{fold_num}.xlsx')
df_cluster = df_cluster.loc[df_cluster['Dataset']=='train']
df_cluster.reset_index(drop=True, inplace=True)
custom_loss_weight = []
total_sample = df_cluster.shape[0]
for x in graph_label_dict.keys():
    df_x = df_cluster.loc[df_cluster['Activate']==x]
    df_x.reset_index(drop=True, inplace=True)
    x_sample = df_x.shape[0]
    custom_loss_weight.append(total_sample/x_sample)

assert len(custom_loss_weight) == label_dim_train, 'Wrong custom loss weight, please check what happen.'

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    prog_args.aggregator_type,
    assign_dim,
    prog_args.pool_ratio,
    custom_loss_weight
)
print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()

===============================fold num is : 1, family type is : GTB================================
=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6_GTB_fold1', pool_ratio=0.1, num_pool=1, cuda=1, lr=1.0, clip=inf,
batch_size=128, epoch=10, n_worker=10, gc_per_block=3, aggregator_type='meanpool', dropout=0.0,
method='diffpool', bn=True, bias=True, save_dir='./model_param', load_epoch=-1, data_mode='default',
linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type='GTB')
================================================加载数据================================================
++++++++++ STATISTICS ABOUT THE DATASET ++++++++++
dataset feature dimension is 7
dataset label dimension is 10
the max num node is 655
number of graphs is 19896
++++++++++MODEL STATISTICS++++++++
model hidden dim is 64
model embedding dim for graph instance embedding 64
initial batched pool graph dim is 65
model init finished


# 训练

In [4]:
logger = train(
    train_dataloader, model, prog_args, val_dataset=test_dataloader
)


EPOCH ###### 0 ######
train accuracy for this epoch 0 is 11.01%
loss 145804.6406 with epoch time 81.4381 s & computation time 75.9819 s 

EPOCH ###### 1 ######
train accuracy for this epoch 1 is 9.91%
loss 154359.2969 with epoch time 81.0674 s & computation time 75.5170 s 

EPOCH ###### 2 ######
train accuracy for this epoch 2 is 10.52%
loss 149845.6875 with epoch time 80.4826 s & computation time 74.9664 s 

EPOCH ###### 3 ######
train accuracy for this epoch 3 is 10.79%
loss 152942.3750 with epoch time 80.8492 s & computation time 75.2823 s 

EPOCH ###### 4 ######
train accuracy for this epoch 4 is 10.83%
loss 152739.2969 with epoch time 80.6630 s & computation time 75.0903 s 

EPOCH ###### 5 ######
train accuracy for this epoch 5 is 11.51%
loss 151579.0625 with epoch time 81.7435 s & computation time 76.2290 s 

EPOCH ###### 6 ######
train accuracy for this epoch 6 is 16.44%
loss 153018.5938 with epoch time 81.9434 s & computation time 76.4184 s 

EPOCH ###### 7 ######
train accura

# 评估

In [4]:
def evaluate(dataloader, model, label_dim=None):
    """
    evaluate function
    """
    model.eval()
    correct_label = 0
    with torch.no_grad():
        val_pred_indi = torch.tensor([], device='cuda')
        val_label_indi = torch.tensor([], device='cuda')
        for batch_idx, (batch_graph, graph_labels) in enumerate(dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()
            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            val_pred_indi = torch.cat((val_pred_indi, indi), dim=0)
            val_label_indi = torch.cat((val_label_indi, graph_labels), dim=0)
            correct = torch.sum(indi == graph_labels)
            correct_label += correct.item()
        val_f1_score = f1_score(val_pred_indi.cpu(), val_label_indi.cpu(), average='macro')

        # 用来解决可能缺少类的问题
        addition_list = torch.tensor([x for x in range(label_dim)], device='cuda')
        val_pred_indi = torch.cat((val_pred_indi, addition_list), dim=0)
        val_label_indi = torch.cat((val_label_indi, addition_list), dim=0)

        val_f1_score_classes = f1_score(val_pred_indi.cpu(), val_label_indi.cpu(), average=None)
    result = correct_label / len(dataloader.dataset)
    return result, val_f1_score, val_f1_score_classes


In [ ]:
# 先过一遍train_dataloader，让模型中的一些参数先初始化一下
model.train()
for batch_idx, (batch_graph, graph_labels) in enumerate(train_dataloader):
    for key, value in batch_graph.ndata.items():
        batch_graph.ndata[key] = value.float()
    graph_labels = graph_labels.long()
    if torch.cuda.is_available():
        batch_graph = batch_graph.to(torch.cuda.current_device())
        graph_labels = graph_labels.cuda()
    ypred = model(batch_graph)
    loss = model.loss(ypred, graph_labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=prog_args.clip)
    model.zero_grad()


f_validation_log = open(prog_args.save_dir + "/" + prog_args.dataset + "/validation_log.csv", 'w')

def write_log_columns(temp_line, line_flag, data_log_type, label_dim_train):
    temp_line = temp_line + f',{data_log_type}_accuracy,{data_log_type}_f1_score'
    line_flag += 2
    for x in range(label_dim_train):
        temp_line = temp_line + f',{data_log_type}_f1_score_class_{x}'
        line_flag += 1
    return temp_line, line_flag

temp_line = 'epoch'
line_flag = 1
temp_line, line_flag = write_log_columns(temp_line, line_flag, 'validation', label_dim_train)
temp_line, line_flag = write_log_columns(temp_line, line_flag, 'test', label_dim_train)
temp_line, line_flag = write_log_columns(temp_line, line_flag, 'nova', label_dim_train)
temp_line = temp_line + '\n'
f_validation_log.write(temp_line)

def evaluate_dataloader(dataloader, model, label_dim_train=None, temp_line=None, temp_line_flag=None):
    result, val_f1_score, val_f1_score_classes = evaluate(dataloader, model, label_dim=label_dim_train)
    temp_line = temp_line + f',{result* 100},{val_f1_score}'
    temp_line_flag += 2
    for x in val_f1_score_classes:
        temp_line = temp_line + f',{x}'
        temp_line_flag += 1
    return temp_line, temp_line_flag

# for epoch in range(0,prog_args.epoch):
for epoch in range(0,10):
    begin_time = time.time()
    print("\nEPOCH ###### {} ######".format(epoch))
    if epoch is not None and prog_args.save_dir is not None:
        model.load_state_dict(
            torch.load(
                prog_args.save_dir
                + "/"
                + prog_args.dataset
                + "/model.iter-"
                + "{:04d}".format(epoch), weights_only=True
            )
        )
    temp_line = f'{epoch}'
    temp_line_flag = 1
    temp_line, temp_line_flag = evaluate_dataloader(validation_dataloader, model, label_dim_train=label_dim_train, temp_line=temp_line, temp_line_flag=temp_line_flag)
    temp_line, temp_line_flag = evaluate_dataloader(test_dataloader, model, label_dim_train=label_dim_train, temp_line=temp_line, temp_line_flag=temp_line_flag)
    temp_line, temp_line_flag = evaluate_dataloader(nova_dataloader, model, label_dim_train=label_dim_train, temp_line=temp_line, temp_line_flag=temp_line_flag)
    temp_line = temp_line + '\n'
    assert temp_line_flag == line_flag, 'Wrong log line number, please check what happen.'
    f_validation_log.write(temp_line)

    elapsed_time = time.time() - begin_time
    print("epoch {:.4f} with epoch time {:.4f} s".format(epoch, elapsed_time))

f_validation_log.close()


EPOCH ###### 0 ######
epoch 0.0000 with epoch time 6.9561 s

EPOCH ###### 1 ######
epoch 1.0000 with epoch time 6.8698 s

EPOCH ###### 2 ######
epoch 2.0000 with epoch time 6.8842 s

EPOCH ###### 3 ######
epoch 3.0000 with epoch time 6.8870 s

EPOCH ###### 4 ######
epoch 4.0000 with epoch time 6.8729 s

EPOCH ###### 5 ######
epoch 5.0000 with epoch time 6.8466 s

EPOCH ###### 6 ######
epoch 6.0000 with epoch time 6.9638 s

EPOCH ###### 7 ######
epoch 7.0000 with epoch time 6.8392 s

EPOCH ###### 8 ######
epoch 8.0000 with epoch time 6.8082 s

EPOCH ###### 9 ######
epoch 9.0000 with epoch time 6.8552 s
